# SynCOM: Synthetic Corona Outflow Model (v3-periodic_blobs)

This notebook runs the Python version of SynCOM step by step. Each function is introduced by a **What it does / expected behaviour** cell: inputs, outputs and what you should see.

| section | function |
|---|---|
| 1 | `default_syncom_prams` |
| 2 | `syncom_load` |
| 3 | `syncom_blob_table` |
| 4 | `gaussian_profile`, `gaussian_wave` |
| 5 | `syncom_image` |
| 6 | `syncom_core` |
| 7 | `write_fits_frames` |
| 8 | `save_syncom_movie` |
| 9 | `polar_to_rect`, `plot_rect_image`, `save_rect_movie` |

**What's new in v3-periodic_blobs**
- **Periodic blobs.** Every blob is launched again each time its period is up, before and during the whole run. The field is equally full at every frame, from the first to the last.
- **Launch radius.** A blob begins from its **centre**. It is launched with its centre 6 radial widths below 5 R☉ (the bottom of the polar image), behind the occulter, and slides into the field fully formed. The mean brightness is the same at every radius.
- **Speed.** Everything that doesn't change with time (the blob table and the angular profiles) is computed once per run.
- **FITS on demand.** `syncom_core` no longer writes a FITS file for every frame. `write_fits_frames` writes only the frames you choose.

The tests and the evidence for these changes are in `syncom_notebook_tests.ipynb`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time
from datetime import date

from punchbowl.data.visualize import cmap_punch

from syncom_prams import default_syncom_prams
from syncom_load import syncom_load, syncom_blob_table
from gaussian_function import gaussian_wave
from syncom_image import syncom_image, angular_profiles
from syncom_core import syncom_core
from syncom_fits import write_fits_frames
from syncom_path import syncom_output_path, syncom_run_dir, syncom_run_name
from syncom_movie import save_syncom_movie, save_rect_movie, plot_rect_image, plot_syncom_image
from polar_to_rect import polar_to_rect

# 1. Parameters: `default_syncom_prams`

## What it does
`default_syncom_prams` (in `syncom_prams.py`) returns the dictionary of settings that every other SynCOM function reads. If `SAVEPRAMS` is given, it also saves them to a `.npy` file.

| key | default | meaning |
|---|---|---|
| `syncom_n_blobs` | 1000 | number of blobs (independent outflow sources) |
| `syncom_nx` | 3600 | pixels in position angle (0.1° each) |
| `syncom_ny` | 659 | pixels in radius |
| `syncom_pixel` | 0.0141876 R☉ | radial pixel size |
| `syncom_cadence` | 300 s | time between two frames |
| `syncom_initial_R_sun` | 5.0 R☉ | radius where blob centres enter the field (the bottom of the polar image) |
| `syncom_scale_factor` | 2.0 | multiplies every blob size |
| `syncom_acc`, `syncom_noise_level`, `syncom_max_intensity` | 0, 0, 1e-14 | stored for compatibility with the IDL version; not used by the current image code |

## Expected behaviour
- The function returns a plain `dict` and prints where the parameters were saved.
- `DATE` changes only the labels (`Label`, `syncomfile`); it does not change the simulation.
- Every later cell takes its grid size, cadence and pixel size from this dictionary, so changing a value here changes the whole run.


In [ ]:
params = default_syncom_prams(SYNCOM_N_BLOBS=5000,DATE="2023-04-11", SAVEPRAMS="syncom_params.npy")

params

In [ ]:
n_blobs = params["syncom_n_blobs"]
nx = params["syncom_nx"]
ny = params["syncom_ny"]
scale_factor = params["syncom_scale_factor"]
pixel_size = params["syncom_pixel"]
cadence = params["syncom_cadence"]
Rsun2Km = 696000.0
spatial_scale = Rsun2Km * pixel_size

# 2. Blob properties: `syncom_load`

## What it does
`syncom_load(params, simple=True)` (in `syncom_load.py`) creates `syncom_n_blobs` blobs and gives each one:
- a **position angle** $\psi$, drawn at random (`generate_blob_positions`)
- a **speed** $v$, **period** $P$ and **size** $L$, read from profiles that depend only on the position angle (`load_velocity_profile`, `assign_blob_properties`)

The blobs are sorted from fastest to slowest and returned as `outarray` (one row per blob):

| column | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| content | launch radius [R☉] | φ [rad] | θ [rad] | group counter (not used by the images) | period [s] | speed [km/s] | size | ψ [0.1° pixels] |

## Expected behaviour
- `outarray.shape == (n_blobs, 8)`.
- The results are reproducible, because of the fixed random seeds (22 for the profiles, 123 for the positions).
- With `simple=True`: speeds are 150–350 km/s, periods are 5400–10800 s (1.5–3 h), and sizes are 0.1–1.1.
- With `simple=False`: the profiles are read from `velocity.npy` and `frequency.npy`, which must be in the working directory.


In [ ]:
# Load with sinusoidal velocity profile
load_struc = syncom_load(params, simple=True)
outarray = load_struc["outarray"]

# Check shape
print(outarray.shape)

# 3. The blob table: `syncom_blob_table`

## What it does
`syncom_blob_table(params, load_struc)` (in `syncom_load.py`) converts each blob's properties from physical units to the image's units, **once per run**. It's a small table (one entry per blob) that `syncom_core` and `syncom_image` read at every time step, like a database:

| key | formula | unit |
|---|---|---|
| `psi` | position angle | px of 0.1° |
| `r0` | $(R_\text{enter} - 5\,R_\odot)/\Delta r - 12L$ | px from the bottom of the image: where the blob's **centre** is at launch (negative: below the image, behind the occulter) |
| `v` | $v\,\Delta t / (R_\odot\,\Delta r)$ | px/frame |
| `period` | $P / \Delta t$ | frames |
| `L` | $\text{scale}\cdot s / 0.1$ | px (angular width; radial width $2L$) |
| `time_0` | uniform 1–201, seed 22 | frames |

## Expected behaviour
- `r0 = -12L` for every blob: each blob is launched with its **centre** 6 radial widths below the bottom row, behind the occulter. In R☉ that's 1.3–4.7 R☉ (2.9 R☉ for a typical blob). Its centre then crosses 5 R☉ with the blob fully formed.
- `v` is 4.6–10.6 px/frame and `period` is 18–36 frames for the default settings.
- The same `params` and `load_struc` always give the same table.


In [ ]:
# Blob table: every blob in pixels and frames, computed once
blobs = syncom_blob_table(params, load_struc)

for key, values in blobs.items():
    print(f"{key:7s} min {values.min():9.3f}   max {values.max():9.3f}")

# 4. The blob shape: `gaussian_profile` and `gaussian_wave`

## What it does
`gaussian_function.py` provides two functions:
- `gaussian_profile(npixel, avr, st_dev, periodic=False)` is a 1D Gaussian of peak 1 centred at `avr` with width `st_dev` (in pixels). It's vectorised: give arrays of centres and widths and it returns one profile per blob. With `periodic=True` the axis wraps around.
- `gaussian_wave(npixel, avr, st_dev)` is a single 1D or 2D blob. In 2D it's the outer product of a **periodic** profile along position angle and a **non-periodic** profile along radius.

## Expected behaviour
- Peak value 1 at the centre. At ±1 width it's $e^{-1/2} \approx 0.61$.
- Along position angle, a blob centred near 359.9° continues at 0°, with no cut at the edge.
- Along radius, a blob near the top of the image is cut off at the edge (material leaves the field).
- The next cell draws one 2D blob at the centre of the grid.


In [ ]:
# Parameters for 2D Gaussian blob
npixel = [3600, 659]         # Image dimensions: [position angle, radial distance]
center = [1800, 300]          # Center position of the Gaussian blob
st_dev = [50, 50]            # Standard deviation in each direction

# Generate the 2D Gaussian profile
blob = gaussian_wave(npixel, center, st_dev)

# Plot the result
plt.figure(figsize=(10, 4))
plt.imshow(blob.T, origin='lower', aspect='auto', cmap=cmap_punch)
plt.colorbar(label='Amplitude')
plt.title('2D Gaussian Propagating Blob')
plt.xlabel('Position Angle [pixels]')
plt.ylabel('Radial Distance [pixels]')
plt.tight_layout()
plt.show()

# 5. One synthetic image: `syncom_image`

## What it does (v3-periodic_blobs)
`syncom_image(params, blobs, time_t, gx=None)` (in `syncom_image.py`) builds the image at time step `time_t`, of shape `(NX, NY)`:
1. Every blob is **launched once per period**, at $\tau = \ldots, -P, 0, P, 2P, \ldots$ on its own clock $\tau = t + t_0$. It was already launching before the first frame, and it keeps launching until the last one.
2. Each launch begins with its **centre 6 radial widths ($12L$) below 5 R☉**, behind the occulter. It moves outward at the blob's speed $v$, enters the image fully formed, and leaves at the top.
3. Each launch is an angular Gaussian (width $L$, periodic) times a radial Gaussian (width $2L$).
4. Only launches that have **already happened** and are **still inside the image** are drawn (`launch_range`).
5. All blobs are summed, as one matrix product per chunk of 500 blobs. `gx`, the angular part, can be passed in so it's computed once per run.

Nothing is written to disk. FITS files are written separately with `write_fits_frames` (section 7).

## Expected behaviour
- At every position angle with a blob there is a **train of blobs, $v P$ pixels apart**, sliding out from under the occulter.
- A new blob centre crosses 5 R☉ every $P$ frames, for every frame of the run. With $P$ = 1 h at a 300 s cadence, that's every 12 frames.
- Blobs come out of the occulter fully formed. The mean brightness is the same at every radius from 5 to 14.3 R☉, with no dim ring at the occulter edge (`syncom_notebook_tests.ipynb`, section 3).
- The field is full from the first frame. In this notebook's run (10000 blobs), the number of launches in the field is 39111–39311 at every frame from 0 to 499, a spread of 0.5% (`syncom_notebook_tests.ipynb`, section 2, shows the same for 2000 blobs).
- The next cell builds one frame and plots it in polar coordinates.


In [ ]:
# One frame (time step 100) with the 5000 blobs loaded above; nothing is written to disk
gx = angular_profiles(params, blobs)                    # angular part of every blob, computed once
img = syncom_image(params, blobs, time_t=100, gx=gx)    # [NX, NY] = [position angle, radius] in pixels

print(img.shape, f"min {img.min():.2f}   max {img.max():.2f}")
plot_syncom_image(img, title="syncom_image: time step 100, 5000 blobs (axes in pixels)")

## Run settings
The next cell sets the run: number of blobs, scale factor, version label and time steps (`time0` to `time_t - 1`). The version label is part of every output file name, so a new label never overwrites an earlier run.


In [ ]:
NY = 7779
PIXEL = 0.0225
N_BLOBS = 1000
scaleFactor = 1.0
dia = date.today().isoformat()        # today's date as YYYY-MM-DD
syncom_version = "v3.2-planet_body"  # label for output files (new label -> earlier runs are not overwritten)
time0, time_t = 0, 100                # time steps time0 ... time_t-1

In [ ]:
RUN_NAME = syncom_run_name(syncom_version, N_BLOBS, scaleFactor)   # prefix of every file of this run
print("Run folder:", syncom_run_dir(RUN_NAME).resolve())

# 6. The full simulation: `syncom_core`

## What it does (v3-periodic_blobs)
`syncom_core(N_BLOBS, time0, time_t, ...)` runs the whole simulation:
1. loads the blobs (`syncom_load`) unless `LoadStruc` is given
2. builds the blob table **once** (`syncom_blob_table`) and the angular profiles **once** (`angular_profiles`). These don't change with time.
3. calls `syncom_image` for $t = $ `time0` … `time_t - 1` and stacks the frames into a cube `[NX, NY, NT]`
4. with `save=True`, saves the cube as a compressed `.npz` (float64, as before)

No FITS files are written here. Choose which frames to write with `write_fits_frames` (section 7).

## Expected behaviour
- It prints where the parameters and the cube were saved, and the total runtime.
- **Expected run times.** Simulating 300 frames (`syncom_image`, Apple M5):

| blobs \ scale factor | 1.0 | 2.0 | 3.0 | 4.0 |
|---|---|---|---|---|
| 1000 | 11 s | 14 s | 14 s | 16 s |
| 2000 | 20 s | 22 s | 26 s | 28 s |
| 5000 | 47 s | 54 s | 62 s | 70 s |
| 10000 | 96 s | 109 s | 132 s | 150 s |

  - Time grows linearly with the number of frames: 500 frames take 5/3 as long.
  - Saving the cube (`save=True`) adds about 2 min per 300 frames (117 s for 5.3 GB), for any number of blobs. The two movies add about 50 s per 300 frames.
  - A larger scale factor costs 40–56% more from 1.0 to 4.0, because more launches are drawn per frame: each blob is launched L$ below the field and drawn until it is L$ above it. One frame takes about 7–8 µs per launch drawn.
  - Examples: 2000 blobs at scale 2 for 300 frames take about 2.5 min in total; 10000 blobs at scale 1 for 500 frames take about 6–7 min.
- The cube holds 3600 × 659 × 500 float64 values: 9.5 GB in memory, 8.9 GB as a compressed `.npz`.
- `N_BLOBS` must equal `params["syncom_n_blobs"]`, otherwise a `ValueError` explains the mismatch.
- Every frame, from the first to the last, shows trains of blobs sliding out from under the occulter at 5 R☉. All frames are equally full.


In [ ]:
# Step 1: Load SynCOM parameters
# params = default_syncom_prams(SYNCOM_N_BLOBS=N_BLOBS, DATE=dia,
#                               SAVEPRAMS=syncom_output_path(RUN_NAME, "run", "params.npy"))
params = default_syncom_prams(SYNCOM_NY = NY, SYNCOM_N_BLOBS=N_BLOBS, DATE=dia, SYNCOM_PIXEL=PIXEL,
                              SAVEPRAMS=syncom_output_path(RUN_NAME, "run", "params.npy"))

cube = syncom_core(
        N_BLOBS,            # Number of blobs
        time0=time0,             # Starting time index
        time_t=time_t,           # Ending time index (exclusive)
        syncom_version=syncom_version,
        scale_factor=scaleFactor,        # Optional override for blob size scaling
        simple=True,             # True -> sinusoidal 250 + 100 cos(2 theta) km/s; False -> velocity.npy
        save=True,               # Save the result as .npz
        params=params            # Parameter dictionary
    )

# 7. FITS frames on demand: `write_fits_frames`

## What it does
`write_fits_frames(cube, run_name, frames=None, time0=0, params=None)` (in `syncom_fits.py`) writes **only the time steps you choose** as FITS files, one file per time step:

```
SynCOM_runs/<run_name>/FITS/<run_name>_<t:04d>.fts      image shape (NY, NX) = (radius, position angle)
```

- `frames=range(0, 30)` writes the first 30 time steps. `frames=[0, 100, 200]` writes three. `frames=None` writes all of them.
- With `params`, the header also stores the cadence, the time since step 0, the pixel size and the number of blobs.
- `load_fits_cube(fits_dir, run_name, time0, time_t, nx, ny)` reads a range of them back into a cube.

## Expected behaviour
- Float64, uncompressed, so the data reads back **bit for bit** (any FITS reader: astropy, IDL `readfits`, FlowTrack).
- One file is about 19 MB. The next cell writes the 30-frame sample (≈ 570 MB).
- Asking for a time step outside the cube raises an `IndexError`.


In [ ]:
# FITS files only for the frames we need: the 30-frame sample, time steps 0-29
# (float64, uncompressed, one ~19 MB file per time step)
# fits_files = write_fits_frames(cube, RUN_NAME, frames=range(0, 30), time0=time0, params=params)

## Time stamps
`START_TIME` is the UTC time given to time step `time0`. Frame $k$ is labelled $t_\text{start} + k\,\Delta t$ in the movies. It only affects the labels.


In [ ]:
START_TIME = Time(f"{dia}T00:00:00", scale="utc")   # UTC time of cube frame 0 (time step time0) - edit as needed

# 8. Polar movie: `save_syncom_movie`

## What it does
`save_syncom_movie(cube, filename, start_time, fps, cadence, title)` (in `syncom_movie.py`) writes the polar cube as an MP4. Position angle runs along x, radius along y, and every frame uses the same colour scale. It needs `ffmpeg`.

## Expected behaviour
- `SynCOM_runs/<run>/movies/<run>_polar_movie.mp4`, about 2.6 MB for this run's 500 frames.
- Each frame's title shows the run label, the UTC time and the frame number.
- Blobs appear as bright patches moving upward. Faster sectors move visibly faster.


In [ ]:
scaleFactor=2.0

# Save the simulated cube as an MP4 movie
save_syncom_movie(
    cube,
    filename=syncom_output_path(RUN_NAME, "movies", "polar_movie.mp4"),
    start_time=START_TIME,
    fps=10,                              # 20 frames -> 4 s video
    cadence=params["syncom_cadence"],   # show elapsed time in the title
    title=RUN_NAME,
    # extent=[0, 360, 0, params["syncom_ny"] * params["syncom_pixel"]],  # physical axes
)

# 9. From polar to Cartesian: `polar_to_rect` and `plot_rect_image`

## What they do
- `polar_to_rect(img_p, phi_range, rho_range, n=1024)` (in `polar_to_rect.py`) resamples a polar frame $(\psi, r)$ onto an $n \times n$ Cartesian grid $(x, y)$ in R☉, the way the corona looks in a coronagraph image. Pixels inside 5 R☉ (the occulter) and outside the field are set to 0.
- `plot_rect_image(rect_img, ...)` displays it with the axes in R☉.

## Expected behaviour
- A ring from 5 R☉ to 14.3 R☉, with a black disk in the middle and black corners.
- Blobs look like short radial streaks pointing away from the Sun. They get wider in angle further out, because one degree spans more distance at larger radius.


In [ ]:
from syncom_prams import POLAR_PHI_RANGE, polar_rho_range

# Radial range of the polar frames, computed from this run's pixel size (the one the simulation used)
POLAR_RHO_RANGE = polar_rho_range(params)     # [Rsun] (bottom row, top row) = (5.0, 14.335)
print("POLAR_PHI_RANGE =", POLAR_PHI_RANGE, "deg;  POLAR_RHO_RANGE =", tuple(round(r, 3) for r in POLAR_RHO_RANGE), "Rsun")

In [ ]:
# Polar -> Cartesian for one time step of the cube.
# Use the cube directly: plot_syncom_image() only draws and returns None, so `img = plot_syncom_image(...)` sets img to None.
frame = 99                                   # time step to show (cube runs from time0 = 0)
rect_img = polar_to_rect(cube[:, :, frame], POLAR_PHI_RANGE, POLAR_RHO_RANGE, n=1024)

The next cell plots the Cartesian frame. The colour bar is in arbitrary brightness units: the peak of one isolated blob is 1, and overlapping blobs add up.


In [ ]:
plot_rect_image(rect_img, rho_max=POLAR_RHO_RANGE[1], title=f"Processed frame, t = {frame} (Cartesian)", cmap=cmap_punch,
                label="Brightness [MSB]")

## ✅ What changed compared with v2-fixed

In v2-fixed this same frame ($t = 100$) was already sparse and dim, and the field kept emptying until it was almost black at the last frame. **The blobs ran out before the end of the simulation**, because each blob was a fixed train of 7 copies that was never renewed.

In v3-periodic_blobs every blob is launched again each time its period is up, beginning with its **centre six radial widths below 5 R☉**, behind the occulter:
- The frame above is filled evenly from the occulter edge (5 R☉) out to 14.3 R☉. Blobs slide out from under the occulter fully formed, so there is no dim ring at its edge: the mean brightness is the same at every radius.
- With 10000 blobs, the number of launches in the field is 39111–39311 at **every** frame from 0 to 499. With the v2 model, 2000 blobs fell from 3425 copies in the field to 3 (`syncom_notebook_tests.ipynb`, section 2). There is no spin-up, because every source was already launching before the first frame.
- The slow-wind sectors (≈90° and 270°) look brightest. Launches there are only $\lambda = vP \approx 1.2$–$2.3\,R_\odot$ apart, so they overlap more than in the fast sectors ($2.7$–$5.4\,R_\odot$).

The evidence (counts, brightness against radius, radius–time diagrams, recovered speeds and frequencies) is in `syncom_notebook_tests.ipynb`.

## Cartesian movie: `save_rect_movie`

### What it does
`save_rect_movie(cube, filename, start_time, title, fps)` converts every frame with `polar_to_rect` and writes an MP4. Every frame uses the same colour scale, and the occulter and corners are black.

### Expected behaviour
- `SynCOM_runs/<run>/movies/<run>_rect_movie.mp4`, about 4.4 MB for this run's 500 frames.
- It looks like a coronagraph movie: blobs stream out from under the occulter in every direction, faster in the fast sectors, and the field is equally full from the first frame to the last.


In [ ]:
rect_title = f"{RUN_NAME}_rect"
save_rect_movie(
    cube,
    filename=syncom_output_path(RUN_NAME, "movies", "rect_movie.mp4"),
    start_time=START_TIME,
    title=rect_title,
    fps=10,
    rho_range=POLAR_RHO_RANGE,   # same radial range as the frame above
    # frames=range(0, 100),   # only part of the cube
)

## Conclusion
In this movie the corona **is equally full from the first frame to the last**: every blob reappears each time its period is up, launched behind the occulter, so the fix for v2-fixed works and there is no dim ring at 5 R☉. The 30-frame FITS sample (time steps 0–29) was written in section 7. The checks of the periodic behaviour, the brightness against radius and the frequencies are in `syncom_notebook_tests.ipynb`.